In [6]:
import cv2
import numpy as np

def preprocess_for_ocr(img_path):
    # 1. Load image
    img = cv2.imread(img_path)
    
    # 2. Grayscale: Removes color noise. OCR models work best on 1-channel images.
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 3. Thresholding (Binarization): Turns the image into pure Black & White.
    # cv2.THRESH_OTSU automatically calculates the best threshold value.
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # 4. Denoising: Removes "salt and pepper" noise (little dots).
    denoised = cv2.fastNlMeansDenoising(thresh, None, 10, 7, 21)
    
    return denoised

# --- Teacher's Note: Why do this? ---
# - Normalization: Ensures pixel values are consistent (0 to 255).
# - Deskewing: If the paper is tilted, the AI struggles. Correcting rotation is vital.
# - Scaling: If the image is tiny, we upscale it. PaddleOCR likes at least 300 DPI.

In [7]:
from paddleocr import PaddleOCR

# 1. Start lean with updated V2/PaddleX compatible arguments
ocr = PaddleOCR(
    use_textline_orientation=True, # Fixed the deprecation warning
    lang='en'
    # Removed use_gpu=False as it causes the ValueError in the new backend
)

# 2. Run the OCR
# Note: make sure 'test.jpg' exists!
result = ocr.ocr("test.jpg", cls=True)

# 3. Safely print results (and grab confidence scores while we're at it)
if result and result[0]:
    for idx, line in enumerate(result[0]):
        # Unpack the PaddleOCR format: [bounding_box, (text, confidence)]
        box, (text, confidence) = line 
        print(f"Line {idx}: {text} (Conf: {confidence:.2f})")
else:
    print("Failed to detect text. Check if test.jpg exists and is readable.")

c:\Users\r1005\Downloads\OCR_pipeline\venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `C:\Users\r1005\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Encountering exception when download model from huggingface: 
Got: ConnectError: [WinError 10054] An existing connection was forcibly closed by the remote host
An error happened while trying to locate the files on the Hub and we cannot find the appropriate snapshot folder for the specified revision on the local disk. Please check your internet connection and try again., will try to download from other model sources: `modelscope`.
U

2026-04-06 17:06:08,154 - modelscope - INFO - Got 5 files, start to download ...
Processing 5 items:   0%|          | 0.00/5.00 [00:00<?, ?it/s]












Processing 5 items:  20%|██        | 1.00/5.00 [00:02<00:11, 2.85s/it]



Processing 5 items:  60%|██████    | 3.00/5.00 [00:03<00:02, 1.03s/it]
Processing 5 items:  80%|████████  | 4.00/5.00 [00:03<00:00, 1.48it/s]











Processing 5 items: 100%|██████████| 5.00/5.00 [00:05<00:00, 1.06s/it]
2026-04-06 17:06:13,501 - modelscope - INFO - Download model 'PaddlePaddle/PP-LCNet_x1_0_doc_ori' successfully.
Creating model: ('UVDoc', None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `C:\Users\r1005\.paddlex\official_models\UVDoc`.
[2026-04-06 17:06:14,535] [    INFO] _client.py:1025 - HTTP Request: GET https://huggingface.co/api/models/PaddlePaddle/UVDoc/revision/main "HTTP/1.1 200 OK"
Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s][2026-04-06 17:06:14,818] [    INFO] _client.py

2026-04-06 17:06:36,044 - modelscope - INFO - Got 5 files, start to download ...
Processing 5 items:   0%|          | 0.00/5.00 [00:00<?, ?it/s]












Processing 5 items:  20%|██        | 1.00/5.00 [00:01<00:07, 1.76s/it]









Processing 5 items:  40%|████      | 2.00/5.00 [00:02<00:03, 1.18s/it]



Processing 5 items:  60%|██████    | 3.00/5.00 [00:02<00:01, 1.33it/s]


Processing 5 items: 100%|██████████| 5.00/5.00 [00:04<00:00, 1.12it/s]
2026-04-06 17:06:40,498 - modelscope - INFO - Download model 'PaddlePaddle/en_PP-OCRv5_mobile_rec' successfully.
C:\Users\r1005\AppData\Local\Temp\ipykernel_3652\2472516992.py:12: DeprecationWarning: Please use `predict` instead.
  result = ocr.ocr("test.jpg", cls=True)


TypeError: PaddleOCR.predict() got an unexpected keyword argument 'cls'

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from datetime import datetime, timezone

# --- SCHEMA 1: NoSQL (The Raw "Review Queue" Model) ---
class MongoDocument(BaseModel):
    filename: str
    raw_text: str
    avg_confidence: float
    # Modern Python standard for UTC time
    extracted_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    status: str = "pending"

# --- SCHEMA 2: SQL (The "Clean Store" Model) ---
class InvoiceSQL(BaseModel):
    invoice_id: str = Field(..., min_length=1)
    date: str
    total_amount: float
    
    # Modern Pydantic V2 standard
    @field_validator('total_amount')
    @classmethod
    def must_be_positive(cls, v: float) -> float:
        if v <= 0:
            raise ValueError('Total amount must be greater than zero')
        return v

C:\Users\r1005\AppData\Local\Temp\ipykernel_3652\1350653742.py:21: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  @validator('total_amount')
